# M1 — Building the 28 warehouse tables

**Goal of this step:** turn 3 messy raw tables into 28 tidy tables that look like a real company's ad data warehouse.

**Analogy:** the raw data is a huge box of shop receipts (one receipt per ad shown). The warehouse is the shelf of
ready-made summary reports, such as "clicks per campaign per day" or "unique people per brand". Many of the reports
look almost the same: hourly vs daily, all traffic vs cleaned traffic, total views vs unique people, live vs final.
**That's on purpose.** Telling those look-alikes apart is exactly what the AI agent will have to learn later.

**Files in this step**
| File | What it is |
|---|---|
| `warehouse/rules.md` | The fixed rules (what "now" means, what counts as a suspicious click, …) |
| `warehouse/build_tables.sql` | One SQL file that builds all 28 tables |
| `warehouse/build.py` | Runs the SQL file and prints row counts + checks |

**Done when:** one command builds every table and the row counts look right ✅

## 1. Connect to the database

**DuckDB** is a database that lives in a single file (`data/ads.duckdb`), so there's no server to set up.
`con.sql("...")` runs a query and shows the result as a table.

In [1]:
import duckdb

con = duckdb.connect("../data/ads.duckdb")
con.sql("SHOW TABLES")

nameVARCHAR
ad_feature
agg_ad_daily
agg_ad_daily_valid
agg_ad_hourly
agg_ad_weekly
agg_advertiser_daily
agg_brand_daily
agg_campaign_daily
agg_campaign_daily_by_audience
agg_campaign_daily_valid


## 2. The 3 raw tables (the "receipts")

| Raw table | One row = | Rows |
|---|---|---|
| `raw_sample` | one ad shown to one user (`clk` = 1 if they clicked) | 26.6 M |
| `ad_feature` | one ad: its campaign, advertiser (`customer`), brand, category, price | 847 k |
| `user_profile` | one user: gender, age group, spending level, … | 1.06 M |

In [2]:
con.sql("SELECT * FROM raw_sample LIMIT 5")

user_idBIGINT,time_stampBIGINT,adgroup_idBIGINT,pidVARCHAR,nonclkBIGINT,clkBIGINT
581738,1494137644,1,430548_1007,1,0
449818,1494638778,3,430548_1007,1,0
914836,1494650879,4,430548_1007,1,0
914836,1494651029,5,430548_1007,1,0
399907,1494302958,8,430548_1007,1,0


In [3]:
con.sql("SELECT * FROM ad_feature LIMIT 5")

adgroup_idBIGINT,cate_idBIGINT,campaign_idBIGINT,customerBIGINT,brandBIGINT,priceDOUBLE
63133,6406,83237,1,95471,170.0
313401,6406,83237,1,87331,199.0
248909,392,83237,1,32233,38.0
208458,392,83237,1,174374,139.0
110847,7211,135256,2,145952,32.99


In [4]:
con.sql("SELECT * FROM user_profile LIMIT 5")

user_idBIGINT,cms_segidBIGINT,cms_group_idBIGINT,final_gender_codeBIGINT,age_levelBIGINT,pvalue_levelBIGINT,shopping_levelBIGINT,occupationBIGINT,new_user_class_levelBIGINT
234,0,5,2,5,NULL,3,0,3
523,5,2,2,2,1,3,1,2
612,0,8,1,2,2,3,0,NULL
1670,0,4,2,4,NULL,1,0,NULL
2545,0,10,1,4,NULL,3,0,NULL


## 3. Time: from a big number to China time

`time_stamp` is **Unix seconds** (seconds since 1 Jan 1970). The data comes from China, so we convert it to
China time (UTC+8). The data covers **6 May → 13 May 2017**, and we pretend **"now" = 13 May 2017, 23:59**
(rule 3 in `rules.md`).

In [5]:
con.sql("""
    SELECT
        MIN(time_stamp) AS first_raw,
        MAX(time_stamp) AS last_raw,
        to_timestamp(MIN(time_stamp)) AT TIME ZONE 'Asia/Shanghai' AS first_china_time,
        to_timestamp(MAX(time_stamp)) AT TIME ZONE 'Asia/Shanghai' AS last_china_time
    FROM raw_sample
""")

first_rawBIGINT,last_rawBIGINT,first_china_timeTIMESTAMP,last_china_timeTIMESTAMP
1494000000,1494691186,2017-05-06 00:00:00,2017-05-13 23:59:46


## 4. Build all 28 tables

This runs every statement in `build_tables.sql` and takes about 10–60 seconds.
It's safe to re-run, because each table uses `CREATE OR REPLACE`.

*(This is the same thing `python warehouse/build.py` does from the terminal.)*

In [6]:
sql = open("../warehouse/build_tables.sql").read()
con.execute(sql)
print("done")

done


## 5. The tables, family by family

### 5a. Dimension tables: "who / what" lookups
Same information as the raw tables, but with **clear names** (`customer` → `advertiser_id`, `final_gender_code` → `gender`).

`dim_ad` · `dim_campaign` · `dim_advertiser` · `dim_user` · `dim_placement` · `dim_date`

In [7]:
con.sql("SELECT * FROM dim_ad LIMIT 5")

ad_idBIGINT,campaign_idBIGINT,advertiser_idBIGINT,brand_idBIGINT,category_idBIGINT,priceDOUBLE
63133,83237,1,95471,6406,170.0
313401,83237,1,87331,6406,199.0
248909,83237,1,32233,392,38.0
208458,83237,1,174374,392,139.0
110847,135256,2,145952,7211,32.99


In [8]:
con.sql("SELECT * FROM dim_user LIMIT 5")

user_idBIGINT,genderVARCHAR,age_levelBIGINT,spending_levelBIGINT,shopping_depthBIGINT,is_studentBIGINT,city_levelBIGINT,micro_group_idBIGINT,group_idBIGINT
234,female,5,NULL,3,0,3,0,5
523,female,2,1,3,1,2,5,2
612,male,2,2,3,0,NULL,0,8
1670,female,4,NULL,1,0,NULL,0,4
2545,male,4,NULL,3,0,NULL,0,10


In [9]:
con.sql("SELECT * FROM dim_date ORDER BY date")

dateDATE,weekdayVARCHAR,week_startTIMESTAMP,is_todayBOOLEAN,is_finalBOOLEAN
2017-05-06,Saturday,2017-05-01 00:00:00,False,True
2017-05-07,Sunday,2017-05-01 00:00:00,False,True
2017-05-08,Monday,2017-05-08 00:00:00,False,True
2017-05-09,Tuesday,2017-05-08 00:00:00,False,True
2017-05-10,Wednesday,2017-05-08 00:00:00,False,True
2017-05-11,Thursday,2017-05-08 00:00:00,False,True
2017-05-12,Friday,2017-05-08 00:00:00,False,True
2017-05-13,Saturday,2017-05-08 00:00:00,True,False


### 5b. The event table: one row per ad shown
`fact_ad_events` is `raw_sample` with China time added and the ad details joined in, so the other tables
don't have to repeat that work.

In [10]:
con.sql("SELECT * FROM fact_ad_events LIMIT 5")

event_timeTIMESTAMP,event_dateDATE,event_hourTIMESTAMP,user_idBIGINT,ad_idBIGINT,campaign_idBIGINT,advertiser_idBIGINT,brand_idBIGINT,category_idBIGINT,placement_idVARCHAR,is_clickBIGINT,priceDOUBLE
2017-05-11 16:05:34,2017-05-11,2017-05-11 16:00:00,700823,106061,212462,152554,NULL,6519,430539_1007,0,45.0
2017-05-12 13:29:05,2017-05-12,2017-05-12 13:00:00,450472,106061,212462,152554,NULL,6519,430548_1007,0,45.0
2017-05-13 20:03:29,2017-05-13,2017-05-13 20:00:00,326397,106061,212462,152554,NULL,6519,430539_1007,1,45.0
2017-05-13 15:03:09,2017-05-13,2017-05-13 15:00:00,242034,106061,212462,152554,NULL,6519,430539_1007,1,45.0
2017-05-13 08:45:50,2017-05-13,2017-05-13 08:00:00,1111817,106061,212462,152554,NULL,6519,430548_1007,0,45.0


### 5c. Grain family: the same numbers at different time levels ⚠️ look-alikes
| Table | One row = | Includes today (13 May)? |
|---|---|---|
| `agg_campaign_hourly` | campaign × hour | **yes**, so today is incomplete |
| `agg_campaign_daily` | campaign × day | no, final up to yesterday |
| `agg_campaign_weekly` | campaign × week (Mon start) | no |

The same pattern exists for ads (`agg_ad_*`), plus daily tables for advertiser, brand, category and placement.

Every table has the same metric columns:
- **impressions** = times the ad was shown
- **clicks**
- **ctr** = clicks ÷ impressions
- **est_value** = clicks × price (an *estimate*; the data has no real revenue)

**See the trap:** the same campaign, three different click totals, depending on which table you pick.

In [11]:
con.sql("""
    SELECT 'hourly (incl. today)' AS source, SUM(clicks) AS clicks FROM agg_campaign_hourly WHERE campaign_id = 359520
    UNION ALL
    SELECT 'daily (final)',  SUM(clicks) FROM agg_campaign_daily  WHERE campaign_id = 359520
    UNION ALL
    SELECT 'weekly (final)', SUM(clicks) FROM agg_campaign_weekly WHERE campaign_id = 359520
""")

sourceVARCHAR,clicksHUGEINT
hourly (incl. today),7266
daily (final),6835
weekly (final),6835


Hourly is bigger because it also counts today. Daily and weekly match. The weekly table also tells you how
many days each week really has data for:

In [12]:
con.sql("SELECT * FROM agg_campaign_weekly WHERE campaign_id = 359520")

week_startTIMESTAMP,campaign_idBIGINT,advertiser_idBIGINT,days_in_weekBIGINT,impressionsBIGINT,clicksHUGEINT,ctrDOUBLE,est_valueDOUBLE
2017-05-01 00:00:00,359520,13787,2,43566,2732,0.06270945232520773,343892.45
2017-05-08 00:00:00,359520,13787,5,74882,4103,0.05479287412195187,506221.08999999997


### 5d. Traffic-quality family: all traffic vs cleaned traffic ⚠️ look-alikes
**Rule 12:** if one user clicks **more than 5 times in the same hour**, it looks like a bot or accidental clicking,
so everything that user did in that hour is "suspicious". Those user-hours are listed in `ref_suspicious_user_hours`.

The `*_valid` tables are the same as the normal ones, but with that suspicious traffic removed.

In [13]:
con.sql("""
    SELECT COUNT(*) AS suspicious_user_hours, SUM(clicks_in_hour) AS suspicious_clicks
    FROM ref_suspicious_user_hours
""")

suspicious_user_hoursBIGINT,suspicious_clicksHUGEINT
6794,49840


In [14]:
con.sql("""
    SELECT 'all traffic'   AS source, SUM(clicks) AS clicks, SUM(clicks) / SUM(impressions) AS ctr
    FROM agg_campaign_daily WHERE campaign_id = 359520
    UNION ALL
    SELECT 'valid traffic', SUM(clicks), SUM(clicks) / SUM(impressions)
    FROM agg_campaign_daily_valid WHERE campaign_id = 359520
""")

sourceVARCHAR,clicksHUGEINT,ctrDOUBLE
all traffic,6835,0.05770464676482507
valid traffic,6556,0.05614407686840055


### 5e. Reach family: unique people, not impressions ⚠️ look-alikes
If one person sees an ad 10 times, that's **10 impressions** but **reach = 1**.

Important: unique people **can't be added up across days**, because someone who saw the ad on Monday and
Tuesday would be counted twice. That's why `reach_campaign_total` is its own table.

In [15]:
con.sql("""
    SELECT 'sum of daily reach (WRONG)' AS method, SUM(unique_users) AS people
    FROM reach_campaign_daily WHERE campaign_id = 359520
    UNION ALL
    SELECT 'total reach (RIGHT)', unique_users
    FROM reach_campaign_total WHERE campaign_id = 359520
""")

methodVARCHAR,peopleHUGEINT
sum of daily reach (WRONG),80764
total reach (RIGHT),51949


### 5f. Audience: campaign split by gender and age
Used for follow-up questions like *"now split that by age"*. Users with no profile show as `unknown`.

In [16]:
con.sql("""
    SELECT age_level, SUM(impressions) AS impressions, SUM(clicks) AS clicks, SUM(clicks) / SUM(impressions) AS ctr
    FROM agg_campaign_daily_by_audience
    WHERE campaign_id = 359520
    GROUP BY age_level ORDER BY age_level
""")

age_levelVARCHAR,impressionsHUGEINT,clicksHUGEINT,ctrDOUBLE
0,9,1,0.1111111111111111
1,5496,315,0.05731441048034935
2,34720,2057,0.05924539170506912
3,51025,2914,0.057109260166585005
4,18294,985,0.05384278998578769
5,4517,315,0.06973655080805845
6,393,22,0.05597964376590331
unknown,3994,226,0.05658487731597396


### 5g. Freshness family: live (last 6 hours) vs final ⚠️ look-alikes
`live_*` tables only hold **13 May 18:00 → 23:59**. They're good for "right now" questions, but wrong for
"yesterday" or "this week".

In [17]:
con.sql("SELECT * FROM live_campaign_last6h WHERE campaign_id = 359520")

campaign_idBIGINT,advertiser_idBIGINT,impressionsBIGINT,clicksHUGEINT,ctrDOUBLE,est_valueDOUBLE
359520,13787,3340,155,0.04640718562874251,19632.94


In [18]:
con.sql("SELECT * FROM live_reach_campaign_last6h WHERE campaign_id = 359520")

campaign_idBIGINT,unique_usersBIGINT,unique_clickersBIGINT
359520,2804,155


## 6. Row counts and checks (the "done" test)
Every table should have rows, and the totals must agree with the raw data.

In [19]:
import sys
sys.path.append("../warehouse")
from build import row_counts, checks

row_counts(con)
checks(con)


28 warehouse tables:
  agg_ad_daily                        2,452,793
  agg_ad_daily_valid                  2,439,510
  agg_ad_hourly                      10,793,256
  agg_ad_weekly                       1,167,533
  agg_advertiser_daily                1,058,204
  agg_brand_daily                       463,601
  agg_campaign_daily                  1,601,009
  agg_campaign_daily_by_audience      5,742,256
  agg_campaign_daily_valid            1,595,477
  agg_campaign_hourly                 8,664,659
  agg_campaign_hourly_valid           8,607,078
  agg_campaign_weekly                   657,256
  agg_category_daily                     39,549
  agg_placement_daily                        14
  dim_ad                                846,811
  dim_advertiser                        255,875
  dim_campaign                          423,436
  dim_date                                    8
  dim_placement                               2
  dim_user                            1,061,768
  fact_ad_events  


Checks:
  every raw impression is in fact_ad_events: True
  hourly clicks = all clicks:                True
  daily clicks = all clicks minus today:     True


## 7. Summary: the 28 tables

| Family | Tables | The trap it creates for the AI |
|---|---|---|
| Dimensions (6) | `dim_ad`, `dim_campaign`, `dim_advertiser`, `dim_user`, `dim_placement`, `dim_date` | needed for joins |
| Events (2) | `fact_ad_events`, `ref_suspicious_user_hours` | very big, so slow if used for everything |
| Grain (10) | `agg_ad_hourly/daily/weekly`, `agg_campaign_hourly/daily/weekly`, `agg_advertiser_daily`, `agg_brand_daily`, `agg_category_daily`, `agg_placement_daily` | hourly vs daily vs weekly |
| Traffic quality (3) | `agg_ad_daily_valid`, `agg_campaign_hourly_valid`, `agg_campaign_daily_valid` | all vs cleaned traffic |
| Reach (3) | `reach_campaign_daily`, `reach_campaign_total`, `reach_brand_daily` | views vs unique people |
| Audience (1) | `agg_campaign_daily_by_audience` | follow-up splits |
| Freshness (3) | `live_ad_last6h`, `live_campaign_last6h`, `live_reach_campaign_last6h` | live vs final |

**Next step (M2):** write a short description "card" for each table and put the cards into Qdrant so they can be searched.

In [20]:
con.close()